<a href="https://colab.research.google.com/github/Ameb8/sudoku-ranker/blob/main/SudokuRanker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Sudoku Ranker

This notebook ranks the difficulty of sudoku puzzle based of the human strategies required to solve them. Once the needed strategies have been determined, a feed-forward neural network is used to assign a numerical difficulty rank to the puzzle.

In [31]:
import numpy as np
import pandas as pd

In [10]:
def init_candidates(board):
  candidates = np.ones((9, 9, 9), dtype=bool)

  for row in range(9):
    for col in range(9):
      if board[row, col] != 0:
        update_candidates(candidates, row, col, board[row, col])

  return candidates

### Update Candidates After Value Placement

In [3]:
def update_candidates(candidates, row, col, val):
  """
  Updates possible candidates after placing val at (row, col).

  candidates: 9x9x9 ndarray of booleans
  row, col: int (0-8)
  val: int (1-9)
  """
  digit_idx = val - 1

  # Set only placed digit as possible in this cell
  candidates[row, col, :] = False
  candidates[row, col, digit_idx] = True

  # Eliminate val as candidate from same row and column
  candidates[row, :, digit_idx] = False
  candidates[:, col, digit_idx] = False

  # Eliminate val as candidate from subsquare
  box_row_start = (row // 3) * 3
  box_col_start = (col // 3) * 3
  candidates[box_row_start:box_row_start+3, box_col_start:box_col_start+3, digit_idx] = False

  # Restore True for placed value in its own cell
  candidates[row, col, digit_idx] = True


### Class for Determining All Locations in Subsections Containing Specific Location

In [ ]:
class CellGroup:
  def __init__(self, row, col):
    self.row = [(row, c) for c in range(9)]
    self.col = [(r, col) for r in range(9)]
    box_row = (row // 3) * 3
    box_col = (col // 3) * 3
    self.sqr = [(box_row + r, box_col + c) for r in range(3) for c in range(3)]

# Human Strategies for Solving Sudoku

### Naked Singles

In [4]:
def solve_naked_singles(board, candidates):
  """
  Solves the naked singles strategy.

  board: 9x9 ndarray of ints
  candidates: 9x9x9 ndarray of booleans
  """
  can_fill = []

  for i in range(9): # Iterate rows
    for j in range(9): # Iterate columns
      if board[i, j] != 0:
        continue # Cell filled, continue
      num_candidates = 0
      for k in range(9): # Iterate candidates
        if candidates[i, j, k]:
          num_candidates += 1
          digit_idx = k
      if num_candidates == 1: # Only 1 possibility, save
        can_fill.append((i, j, digit_idx))

  # Fill valid cells
  for i, j, k in can_fill:
    board[i, j] = k + 1
    update_candidates(candidates, i, j, k + 1)

  return len(can_fill)


### Hidden Singles

In [7]:
def solve_hidden_singles(board, candidates):
  """
  Solves the hidden singles strategy.

  board: 9x9 ndarray of ints
  candidates: 9x9x9 ndarray of booleans
  """
  filled = 0

  # Check rows
  for digit in range(9):
    for row in range(9):
      position = [(row, col) for col in range(9) if board[row, col] == 0 and candidates[row, col, digit]]
      if len(position) == 1:
        r, c = position[0]
        board[r, c] = digit + 1
        update_candidates(candidates, r, c, digit + 1)
        filled += 1

  # Check columns
  for digit in range(9):
    for col in range(9):
      position = [(row, col) for row in range(9) if board[row, col] == 0 and candidates[row, col, digit]]
      if len(position) == 1:
        r, c = position[0]
        board[r, c] = digit + 1
        update_candidates(candidates, r, c, digit + 1)
        filled += 1

  # Check boxes
  for digit in range(9):
    for box_row in range(3):
      for box_col in range(3):
        positions = []
        for i in range(3):
          for j in range(3):
            r = box_row * 3 + i
            c = box_col * 3 + j
            if board[r, c] == 0 and candidates[r, c, digit]:
              positions.append((r, c))
        if len(positions) == 1:
          r, c = positions[0]
          board[r, c] = digit + 1
          update_candidates(candidates, r, c, digit + 1)
          filled += 1

  return filled


### Naked Pairs

In [12]:
def solve_naked_pairs(board, candidates):
  """
  Solves puzzle with the naked pairs strategy.

  board: 9x9 ndarray of ints
  candidates: 9x9x9 ndarray of booleans
  """
  total_elims = 0

  for unit_index in range(9):
    row_group = [(unit_index, col) for col in range(9)]
    col_group = [(row, unit_index) for row in range(9)]
    sqr_group = CellGroup(unit_index // 3 * 3, unit_index % 3 * 3).sqr


    total_elims += solve_naked_pairs_group(candidates, row_group)
    total_elims += solve_naked_pairs_group(candidates, col_group)
    total_elims += solve_naked_pairs_group(candidates, sqr_group)

  return total_elims

In [28]:
def solve_naked_pairs_group(candidates, group):
  """
  Solves with the naked pairs strategy for a subsection of board

  candidates: 9x9x9 ndarray of booleans
  group: list of tuples (row, col)
  """

  pair_map = {}
  found_pairs = []
  elims = 0

  # Find naked pairs
  for r, c in group:
    cell_possibilites = []
    for d in range(9):
      if candidates[r, c, d]:
        cell_possibilites.append(d)
    if len(cell_possibilites) == 2:
      if tuple(cell_possibilites) in pair_map:
        found_pairs.append((pair_map[tuple(cell_possibilites)], (r, c)))
      else:
        pair_map[(tuple(cell_possibilites))] = (r, c)

  # Remove found pairs
  for (cell1, cell2) in found_pairs:
    r1, c1 = cell1
    r2, c2 = cell2

    # Get shared candidates
    digits = [d for d in range(9) if candidates[r1, c1, d]]

    for r, c in group:
      if (r, c) != (r1, c1) and (r, c) != (r2, c2):
        for d in digits:
          if candidates[r, c, d]:
            candidates[r, c, d] = False
            elims += 1

  return elims



# Solve Puzzle Tracking Methods Used

In [ ]:
def solve_puzzle(puzzle):
  """
  Solves a sudoku puzzle.

  puzzle: 9x9 ndarray of ints
  """
  candidates = init_candidates(puzzle)
  methods_used = np.zeros(3, dtype=bool)
  num_empty_cells = np.count_nonzero(puzzle == 0)

  while num_empty_cells > 0:
    # Check for naked singles


